# Homework 1: Softmax for MNIST Classification

Build a ten-class linear classifier in NumPy. Complete the four marked `TODO` blocks, run the checks, and use your results in a short PDF report. See `HW1_Softmax_MNIST.tex` for full instructions and grading.

**Setup:** Python 3, NumPy, and Matplotlib. Keep this notebook, `mnist_data_loader.py`, and the supplied `mnist_data/` directory together in `HW1/`. The helper reads the local IDX files; running the notebook does not require internet access. Do not submit the dataset directory.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mnist_data_loader import load_mnist

SEED = 2026

In [ ]:
images_all, labels_all, images_test, labels_test = load_mnist()
assert images_all.shape == (60_000, 28, 28) and images_test.shape == (10_000, 28, 28)
X_all = images_all.reshape(60_000, 784).astype(np.float32) / 255.0
y_all = labels_all
X_test = images_test.reshape(10_000, 784).astype(np.float32) / 255.0
y_test = labels_test

# Split the original training data; do not use the test set for training or selection.
permutation = np.random.default_rng(SEED).permutation(60_000)
train_indices, val_indices = permutation[:50_000], permutation[50_000:]
X_train, y_train = X_all[train_indices], y_all[train_indices]
X_val, y_val = X_all[val_indices], y_all[val_indices]
assert X_train.shape == (50_000, 784) and X_val.shape == (10_000, 784)
print('train / validation / test:', X_train.shape, X_val.shape, X_test.shape)

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, image, label in zip(axes, images_all[:8], labels_all[:8]):
    ax.imshow(image, cmap='gray', vmin=0, vmax=255)
    ax.set_title(str(label))
    ax.axis('off')
plt.tight_layout()
plt.show()

## 1. Model and loss

For a batch $X\in\mathbb{R}^{B\times784}$, scores are $Z=XW+b$ (NumPy broadcasts $b$ across rows). Softmax gives $P_{ik}=\exp(Z_{ik}-m_i)/\sum_j\exp(Z_{ij}-m_i)$, where $m_i=\max_j Z_{ij}$. Minimize

$$L=-\frac{1}{B}\sum_i\log P_{i,y_i}+\frac{\lambda}{2}\lVert W\rVert_F^2.$$

Regularize $W$, not $b$. In the report, derive both gradients and explain why subtracting the row maximum helps. In code, obtain log probabilities from shifted scores and `log(sum(exp(shifted)))`; directly taking `log(probability)` can yield `log(0)` for extreme scores.

In [ ]:
def stable_softmax(logits):
    """Return row-wise probabilities; logits has shape (B, C)."""
    # TODO 1: Subtract each row maximum, exponentiate, and normalize each row.
    # Use broadcasting; no loops over examples or classes.
    raise NotImplementedError('Complete TODO 1: stable_softmax')

In [ ]:
def softmax_loss_and_grad(X, y, W, b, reg):
    """Return scalar loss, dW, db for a nonempty batch.

    X: (B, D), y: (B,), W: (D, C), b: (C,).
    L2 penalty: 0.5 * reg * sum(W ** 2). Do not penalize b.
    """
    batch_size = X.shape[0]
    if batch_size == 0:
        raise ValueError('The batch must be nonempty')
    scores = X @ W + b
    probabilities = stable_softmax(scores)
    # TODO 2: Compute stable mean negative log-likelihood plus L2 penalty;
    # derive dW and db using a (B, C) probabilities-minus-one-hot matrix.
    # Do not modify probabilities in place if you need them later.
    raise NotImplementedError('Complete TODO 2: loss and gradients')

In [ ]:
def predict(X, W, b):
    """Return one predicted integer label per row of X."""
    # TODO 3: Compute scores and select the class with the largest score.
    raise NotImplementedError('Complete TODO 3: predict')

def accuracy(X, y, W, b):
    return float(np.mean(predict(X, W, b) == y))

## 2. Check your implementation

Run this cell after completing TODO 1–3. It tests extreme logits and checks your gradients against finite differences on a small synthetic batch. Loops in this diagnostic checker are allowed; the classifier itself must be vectorized.

In [ ]:
extreme_logits = np.array([[1000., 999., -1000.], [-1000., 0., 1000.]])
p_extreme = stable_softmax(extreme_logits)
assert np.isfinite(p_extreme).all() and np.all(p_extreme >= 0)
assert np.allclose(p_extreme.sum(axis=1), 1.0)
extreme_loss, _, _ = softmax_loss_and_grad(
    np.array([[1000.]]), np.array([1]),
    np.array([[1., -1., 0.]]), np.zeros(3), 0.0)
assert np.isfinite(extreme_loss) and extreme_loss > 1000

check_rng = np.random.default_rng(7)
X_check = check_rng.normal(size=(5, 4))
y_check = np.array([0, 2, 1, 2, 0])
W_check = check_rng.normal(scale=0.1, size=(4, 3))
b_check = check_rng.normal(scale=0.1, size=3)
reg_check = 0.03
loss_check, dW_check, db_check = softmax_loss_and_grad(
    X_check, y_check, W_check, b_check, reg_check)
assert np.isfinite(loss_check)
assert dW_check.shape == W_check.shape and db_check.shape == b_check.shape
assert np.array_equal(predict(X_check, W_check, b_check),
                      np.argmax(X_check @ W_check + b_check, axis=1))

step = 1e-5
numeric_dW = np.zeros_like(W_check)
for index in np.ndindex(W_check.shape):
    W_plus, W_minus = W_check.copy(), W_check.copy()
    W_plus[index] += step
    W_minus[index] -= step
    plus = softmax_loss_and_grad(X_check, y_check, W_plus, b_check, reg_check)[0]
    minus = softmax_loss_and_grad(X_check, y_check, W_minus, b_check, reg_check)[0]
    numeric_dW[index] = (plus - minus) / (2 * step)

numeric_db = np.zeros_like(b_check)
for index in np.ndindex(b_check.shape):
    b_plus, b_minus = b_check.copy(), b_check.copy()
    b_plus[index] += step
    b_minus[index] -= step
    plus = softmax_loss_and_grad(X_check, y_check, W_check, b_plus, reg_check)[0]
    minus = softmax_loss_and_grad(X_check, y_check, W_check, b_minus, reg_check)[0]
    numeric_db[index] = (plus - minus) / (2 * step)

print('Largest absolute gradient errors:',
      np.max(np.abs(dW_check - numeric_dW)), np.max(np.abs(db_check - numeric_db)))
assert np.allclose(dW_check, numeric_dW, atol=1e-6, rtol=1e-5)
assert np.allclose(db_check, numeric_db, atol=1e-6, rtol=1e-5)
print('All checks passed.')

## 3. Mini-batch training

Only the epoch and batch loops are supplied. Use each batch's gradients to update the current parameters. Validation images must never enter a parameter update. The history records full training loss and training/validation accuracy after each epoch.

In [ ]:
def train_softmax(X_train, y_train, X_val, y_val, *, learning_rate, reg,
                  epochs=8, batch_size=256, seed=SEED):
    rng = np.random.default_rng(seed)
    W = np.zeros((X_train.shape[1], 10), dtype=np.float32)
    b = np.zeros(10, dtype=np.float32)
    history = {'train_loss': [], 'train_accuracy': [], 'val_accuracy': []}
    for epoch in range(epochs):
        order = rng.permutation(len(X_train))
        for start in range(0, len(order), batch_size):
            batch_indices = order[start:start + batch_size]
            X_batch, y_batch = X_train[batch_indices], y_train[batch_indices]
            _, dW, db = softmax_loss_and_grad(X_batch, y_batch, W, b, reg)
            # TODO 4: Update W and b using learning_rate and dW/db.
            # Both gradients above used the same old parameters.
            raise NotImplementedError('Complete TODO 4: parameter update')
        train_loss = softmax_loss_and_grad(X_train, y_train, W, b, reg)[0]
        train_acc = accuracy(X_train, y_train, W, b)
        val_acc = accuracy(X_val, y_val, W, b)
        history['train_loss'].append(float(train_loss))
        history['train_accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)
        print(f'epoch {epoch + 1:2d}/{epochs}: loss={train_loss:.4f}, '
              f'train acc={train_acc:.4f}, val acc={val_acc:.4f}')
    return W, b, history

## 4. Compare hyperparameters

The settings below compare two learning rates with fixed regularization and two regularization values with fixed learning rate. You may modify them. Select a model using **validation accuracy only**; do not look at the test set here.

In [ ]:
configs = [
    {'name': 'baseline', 'learning_rate': 0.10, 'reg': 1e-4},
    {'name': 'smaller learning rate', 'learning_rate': 0.03, 'reg': 1e-4},
    {'name': 'stronger regularization', 'learning_rate': 0.10, 'reg': 1e-2},
]
runs = {}
for config in configs:
    print('\nConfiguration:', config['name'])
    W, b, history = train_softmax(
        X_train, y_train, X_val, y_val,
        learning_rate=config['learning_rate'], reg=config['reg'],
        epochs=8, batch_size=256, seed=SEED)
    runs[config['name']] = {'config': config, 'W': W, 'b': b, 'history': history}

print('\nResults (copy to your report):')
print(f"{'configuration':26s} {'lr':>8s} {'lambda':>10s} {'train acc':>12s} {'val acc':>10s}")
for name, run in runs.items():
    cfg, h = run['config'], run['history']
    print(f"{name:26s} {cfg['learning_rate']:8.3g} {cfg['reg']:10.3g} "
          f"{h['train_accuracy'][-1]:12.4f} {h['val_accuracy'][-1]:10.4f}")
best_name = max(runs, key=lambda name: runs[name]['history']['val_accuracy'][-1])
print('Selected by final validation accuracy:', best_name)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, run in runs.items():
    h = run['history']
    epoch_axis = np.arange(1, len(h['train_loss']) + 1)
    axes[0].plot(epoch_axis, h['train_loss'], marker='o', label=name)
    axes[1].plot(epoch_axis, h['train_accuracy'], marker='o', label=f'{name}: train')
    axes[1].plot(epoch_axis, h['val_accuracy'], marker='x', linestyle='--',
                 label=f'{name}: val')
axes[0].set(xlabel='Epoch', ylabel='Training loss', title='Loss')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Accuracy')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Final test evaluation

Run this cell once after selecting the configuration above. Do not change the settings based on the test result.

In [ ]:
best_run = runs[best_name]
test_accuracy = accuracy(X_test, y_test, best_run['W'], best_run['b'])
print(f'Selected configuration: {best_name}')
print(f"Final validation accuracy: {best_run['history']['val_accuracy'][-1]:.4f}")
print(f'Test accuracy: {test_accuracy:.4f}')

## Report checklist

- Derive both gradients, give their shapes, and explain the stability trick.
- Include the results table, training loss curve, and training/validation accuracy curves.
- Compare learning rate and regularization, discuss fit and a limitation of this linear classifier.
- State the selected configuration and its one-time test accuracy. Submit this executed notebook, `mnist_data_loader.py`, and your PDF report. Do not submit `mnist_data/`.